# CPF Week 4 Bridge Capstone — Efficient Markets Hypothesis

## AAPL: Weak-Form Efficiency Assessment

I assess weak-form market efficiency for AAPL using historical daily price data from the TPQ CPF Bridge companion dataset.

My analysis focuses on two empirical diagnostics from the Week 4 EMH material: lag-one return autocorrelation and a variance-ratio test. I use the results to evaluate whether the selected sample provides evidence that is more consistent with or less consistent with weak-form efficiency.

I treat the findings as sample-based evidence rather than as a definitive statement about AAPL or market efficiency in general. I conclude by identifying one extension that could improve the robustness of the assessment.

## Official Capstone Requirements

The official Week 4 Bridge capstone asks me to:

1. Choose a liquid asset or index and gather daily returns for several years.
2. Run two distinct efficiency tests, for example autocorrelation and variance ratio, and summarize the results.
3. Describe whether the findings support or weaken the EMH for this market.
4. Propose one extension that would improve the robustness of the assessment.

## Data and Methodology

I use AAPL end-of-day price data from the TPQ CPF Bridge companion dataset.

I first load the full dataset and then select the AAPL price series. Before calculating returns, I verify that the dates are ordered, the selected series is nonempty, and there are no missing, zero, or negative price observations.

I convert prices to daily log returns using

$$
r_t
=
\ln\left(
\frac{P_t}{P_{t-1}}
\right).
$$

I then evaluate weak-form efficiency using two diagnostics from the Week 4 EMH material:

1. lag-one autocorrelation, which measures linear dependence between consecutive daily returns; and
2. a variance ratio, which compares multi-period return variance with the variance implied by a random-walk benchmark.

I interpret the two results jointly and treat them as sample-specific evidence rather than as a definitive test of market efficiency.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
np.set_printoptions(precision=6, suppress=True)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.6f}"
)

In [3]:
ASSET = "AAPL"

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "yhilpisch/cpfbridge/main/markets/data/eoddata.csv"
)

print("Asset:", ASSET)

Asset: AAPL


In [4]:
prices_raw = pd.read_csv(
    DATA_URL,
    parse_dates=["Date"],
    index_col="Date"
)

In [5]:
print("Shape:", prices_raw.shape)

print(
    "Date range:",
    prices_raw.index.min(),
    "to",
    prices_raw.index.max()
)

print("\nColumns:")
print(prices_raw.columns.tolist())

Shape: (2515, 8)
Date range: 2015-11-30 00:00:00 to 2025-11-28 00:00:00

Columns:
['AAPL', 'NVDA', 'JPM', 'SPY', 'GLD', 'TLT', 'EURUSD', 'BTC-USD']


## Data Selection and Validation

I select AAPL from the full TPQ CPF Bridge dataset and use the complete available sample from November 30, 2015 through November 28, 2025.

Before constructing returns, I inspect the beginning and end of the price series and check its basic data quality. I verify that the observations are ordered chronologically, that the series contains no missing values, and that all recorded prices are strictly positive.

These checks establish that the selected price series can be used directly to construct daily log returns without imputation or other missing-data adjustments.

In [6]:
prices = prices_raw[ASSET].copy()

print("Selected asset:", prices.name)
print("Number of price observations:", len(prices))

print(
    "Date range:",
    prices.index.min(),
    "to",
    prices.index.max()
)

print("Missing values:", prices.isna().sum())
print("Minimum observed price:", prices.min())

print("\nFirst five observations:")
display(prices.head())

print("\nLast five observations:")
display(prices.tail())

Selected asset: AAPL
Number of price observations: 2515
Date range: 2015-11-30 00:00:00 to 2025-11-28 00:00:00
Missing values: 0
Minimum observed price: 20.6041

First five observations:


,AAPL
Date,
2015-11-30,26.673000
2015-12-01,26.456500
2015-12-02,26.217500
2015-12-03,25.974000
2015-12-04,26.837600



Last five observations:


,AAPL
Date,
2025-11-21,271.490000
2025-11-24,275.920000
2025-11-25,276.970000
2025-11-26,277.550000
2025-11-28,278.850000


In [7]:
assert prices.name == ASSET
assert len(prices) > 0
assert prices.index.is_monotonic_increasing
assert not prices.isna().any()
assert np.isfinite(prices.to_numpy()).all()
assert (prices > 0).all()

print("Price data validation: PASS")
print("Number of AAPL price observations:", len(prices))

Price data validation: PASS
Number of AAPL price observations: 2515


## Return Construction

I convert the validated AAPL price series into daily log returns before applying the weak-form efficiency diagnostics.

For each trading day,

$$
r_t
=
\ln\left(
\frac{P_t}{P_{t-1}}
\right).
$$

I use returns rather than price levels because the EMH tests in this capstone examine whether past return information contains predictive information about future returns.

Because each return requires two consecutive price observations, the 2,515 price observations should produce 2,514 daily log-return observations.

In [8]:
returns = np.log(
    prices / prices.shift(1)
).dropna()

print("Number of return observations:", len(returns))

print(
    "Return date range:",
    returns.index.min(),
    "to",
    returns.index.max()
)

print("Missing returns:", returns.isna().sum())

print("\nFirst five return observations:")
display(returns.head())

Number of return observations: 2514
Return date range: 2015-12-01 00:00:00 to 2025-11-28 00:00:00
Missing returns: 0

First five return observations:


,AAPL
Date,
2015-12-01,-0.008150
2015-12-02,-0.009075
2015-12-03,-0.009331
2015-12-04,0.032708
2015-12-07,-0.006321


In [9]:
assert len(returns) == len(prices) - 1
assert returns.index.is_monotonic_increasing
assert not returns.isna().any()
assert np.isfinite(returns.to_numpy()).all()

print("Return data validation: PASS")
print("Number of AAPL return observations:", len(returns))

Return data validation: PASS
Number of AAPL return observations: 2514


## Test 1 — Lag-One Autocorrelation

I first examine whether consecutive AAPL daily log returns display linear dependence.

I calculate the lag-one autocorrelation,

$$
\rho_1
=
\operatorname{Corr}(r_t,r_{t-1}).
$$

Under a strict weak-form benchmark of no linear predictability from the previous day's return, I would expect the lag-one autocorrelation to be close to zero.

A positive value would indicate short-run return continuation, while a negative value would indicate short-run reversal. I interpret the magnitude as sample evidence about linear dependence rather than as proof of an economically exploitable trading opportunity.

In [10]:
acf_1 = returns.autocorr(lag=1)

print(f"Lag-one autocorrelation: {acf_1:.6f}")

Lag-one autocorrelation: -0.056247


### Interpretation

The estimated lag-one autocorrelation for AAPL is

$$
\rho_1 \approx -0.056247.
$$

The negative sign indicates mild short-run reversal: positive returns are followed, on average, by slightly lower returns, and negative returns by slightly higher returns.

The magnitude is small, so the relationship represents only weak linear dependence between consecutive daily returns. This result is somewhat less consistent with a strict weak-form benchmark of zero linear predictability, but it does not by itself establish an economically meaningful inefficiency.

I therefore treat the autocorrelation result as one piece of sample evidence and compare it with a second diagnostic before drawing a broader conclusion.

## Test 2 — Variance Ratio

I next examine whether AAPL return variance scales with the investment horizon in the way implied by a simple random-walk benchmark.

For a horizon of $q$ trading days, I calculate the variance ratio

$$
VR(q)
=
\frac{
\operatorname{Var}\left(r_t^{(q)}\right)
}{
q\,\operatorname{Var}(r_t)
},
$$

where $r_t^{(q)}$ is the sum of $q$ consecutive daily log returns.

Under the random-walk benchmark,

$$
VR(q)\approx 1.
$$

A value below $1$ is associated with negative serial dependence or mean-reverting behavior, while a value above $1$ is associated with positive serial dependence or trend-like behavior.

I use a five-day horizon, $q=5$, and construct non-overlapping five-day return blocks, consistent with the variance-ratio calculation used in the Week 4 EMH notebook.

In [11]:
q = 5

var_1 = returns.var(ddof=1)

n_blocks = len(returns) // q

blocks = (
    returns.iloc[:n_blocks * q]
    .to_numpy()
    .reshape(n_blocks, q)
)

q_period_returns = blocks.sum(axis=1)

var_q = np.var(q_period_returns, ddof=1)

vr_5 = var_q / (q * var_1)

print(f"Variance ratio, q={q}: {vr_5:.6f}")
print("Number of non-overlapping blocks:", n_blocks)

Variance ratio, q=5: 0.829321
Number of non-overlapping blocks: 502


### Interpretation

At the five-day horizon, the estimated variance ratio is

$$
VR(5)\approx0.829321.
$$

This value is below the random-walk benchmark of $1$. Over the selected sample, five-day AAPL return variance was therefore lower than the variance implied by scaling the one-day return variance proportionally by five.

Within the variance-ratio framework, this pattern is consistent with some negative serial dependence or mean-reverting behavior in AAPL returns over the five-day horizon.

The result points away from the strict random-walk benchmark, but I do not interpret the magnitude alone as a formal rejection of weak-form efficiency because this calculation does not attach a standard error or p-value to the variance ratio. I use it as a second piece of sample evidence alongside the lag-one autocorrelation result.

## Overall EMH Assessment

The two diagnostics point in the same general direction.

The lag-one autocorrelation is negative,

$$
\rho_1 \approx -0.056247,
$$

which indicates mild short-run reversal in consecutive AAPL daily returns.

The five-day variance ratio is also below the random-walk benchmark,

$$
VR(5)\approx0.829321<1,
$$

which is consistent with negative serial dependence or mean-reverting behavior over the five-day horizon.

Taken together, these results provide some sample evidence that weakens a strict weak-form benchmark of completely unpredictable returns and exact random-walk variance scaling for AAPL during the selected period.

However, the evidence is limited. The lag-one autocorrelation is small in magnitude, and the variance-ratio calculation used here does not include a formal significance test. These results therefore do not establish that AAPL was economically inefficient or that the observed dependence could have been exploited profitably.

My conclusion is that the sample contains modest evidence against the strictest weak-form benchmark, but not enough to support a broader claim of persistent or tradeable market inefficiency.

## Robustness Extension

One extension I would use to improve the robustness of the assessment is to repeat the two diagnostics across separate subperiods rather than relying only on the full-sample estimates.

For example, I would divide the AAPL return history into earlier and later sample periods and recompute both the lag-one autocorrelation and the five-day variance ratio in each period.

This would allow me to assess whether the observed negative dependence is stable through time or concentrated in a particular market regime. A relationship that appears only in one subperiod would provide weaker evidence of persistent return predictability than a pattern that remains similar across multiple periods.

I would therefore treat temporal stability as an important robustness check before interpreting the full-sample results as evidence of a persistent weak-form inefficiency.

## Final Validation

I finish by validating the data and calculations used in the assessment.

These checks confirm that the AAPL price and return series are complete and finite, that the lag-one autocorrelation is a valid correlation estimate, and that the five-day variance ratio has been constructed from the intended non-overlapping return blocks.

I also verify that the signs of the two calculated diagnostics are consistent with the interpretations stated above.

In [12]:
# 1. Price-data validation
assert prices.name == ASSET
assert len(prices) > 0
assert prices.index.is_monotonic_increasing
assert not prices.isna().any()
assert np.isfinite(prices.to_numpy()).all()
assert (prices > 0).all()

print("PASS 1 — AAPL price data are valid.")


# 2. Return-data validation
assert len(returns) == len(prices) - 1
assert returns.index.is_monotonic_increasing
assert not returns.isna().any()
assert np.isfinite(returns.to_numpy()).all()

print("PASS 2 — AAPL log returns are valid.")


# 3. Lag-one autocorrelation validation
acf_1_check = returns.autocorr(lag=1)

assert np.isfinite(acf_1)
assert -1.0 <= acf_1 <= 1.0
assert np.isclose(acf_1, acf_1_check)

print("PASS 3 — Lag-one autocorrelation is internally consistent.")


# 4. Variance-ratio construction validation
assert q == 5
assert n_blocks == len(returns) // q
assert len(q_period_returns) == n_blocks
assert np.isfinite(var_1)
assert np.isfinite(var_q)
assert var_1 > 0
assert var_q > 0

vr_5_check = var_q / (q * var_1)

assert np.isclose(vr_5, vr_5_check)
assert vr_5 > 0

print("PASS 4 — Five-day variance ratio is internally consistent.")


# 5. Interpretation validation
assert acf_1 < 0
assert vr_5 < 1

print("PASS 5 — Diagnostic directions match the stated interpretation.")


print("\nALL EMH CAPSTONE VALIDATION CHECKS PASSED.")

PASS 1 — AAPL price data are valid.
PASS 2 — AAPL log returns are valid.
PASS 3 — Lag-one autocorrelation is internally consistent.
PASS 4 — Five-day variance ratio is internally consistent.
PASS 5 — Diagnostic directions match the stated interpretation.

ALL EMH CAPSTONE VALIDATION CHECKS PASSED.
